# Gaussian Naive Bayes Irrigation Need


In [1]:
# Imports and settings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.metrics import balanced_accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.30
FOCUS_CLASS = "High"


In [2]:
# Load only the Kaggle train data; the Kaggle test data is not used for this activity.
df = pd.read_csv("data/train.csv")

X = df.drop(columns=["id", "Irrigation_Need"])
y = df["Irrigation_Need"]

print("Data shape:", df.shape)
print("\nTarget class shares:")
print(y.value_counts(normalize=True).rename("share"))


Data shape: (630000, 21)

Target class shares:
Irrigation_Need
Low       0.587170
Medium    0.379483
High      0.033348
Name: share, dtype: float64


In [3]:
# Train/test split from the Kaggle train data only.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

numeric_cols = X.select_dtypes(exclude="object").columns.tolist()
categorical_cols = X.select_dtypes(include="object").columns.tolist()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))


Train shape: (441000, 19)
Test shape: (189000, 19)
Numeric columns: 11
Categorical columns: 8


In [4]:
# GaussianNB needs dense numeric inputs, so categorical variables are one-hot encoded.
try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", one_hot, categorical_cols),
    ],
    remainder="drop",
)

gnb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", GaussianNB()),
    ]
)

gnb_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [5]:
# Predicted probabilities and the default rule: choose the class with highest probability.
classes = gnb_model.named_steps["classifier"].classes_
probabilities = pd.DataFrame(
    gnb_model.predict_proba(X_test),
    columns=classes,
    index=X_test.index,
)

baseline_pred = classes[np.argmax(probabilities.to_numpy(), axis=1)]

print("Baseline classification report:")
print(classification_report(y_test, baseline_pred, digits=4))
print("Baseline balanced accuracy:", round(balanced_accuracy_score(y_test, baseline_pred), 4))
print("Baseline High F1:", round(f1_score(y_test, baseline_pred, labels=[FOCUS_CLASS], average="macro"), 4))

probabilities.head()


Baseline classification report:


              precision    recall  f1-score   support

        High     0.5303    0.7769    0.6303      6303
         Low     0.8745    0.7985    0.8348    110975
      Medium     0.6993    0.7648    0.7306     71722

    accuracy                         0.7850    189000
   macro avg     0.7014    0.7801    0.7319    189000
weighted avg     0.7966    0.7850    0.7884    189000

Baseline balanced accuracy: 0.7801


Baseline High F1: 0.6303


,High,Low,Medium
38756,1.394922e-16,0.998326,0.001674
373429,8.781340e-01,0.000066,0.121800
69884,4.952730e-02,0.010405,0.940068
575040,7.032352e-03,0.028849,0.964119
433322,5.653342e-09,0.905688,0.094312


In [6]:
# Threshold rule: assign High only when its predicted probability is high enough.
def threshold_predictions(probability_frame, focus_class, threshold):
    class_names = probability_frame.columns.to_numpy()
    proba = probability_frame.to_numpy()

    focus_index = np.where(class_names == focus_class)[0][0]
    other_indices = np.where(class_names != focus_class)[0]
    other_classes = class_names[other_indices]

    use_focus = proba[:, focus_index] >= threshold
    best_other = other_classes[np.argmax(proba[:, other_indices], axis=1)]

    predictions = np.empty(len(probability_frame), dtype=object)
    predictions[use_focus] = focus_class
    predictions[~use_focus] = best_other[~use_focus]
    return predictions


In [7]:
# Pick the threshold that gives the best F1 for the rare High class on the held-out split.
threshold_rows = []

for threshold in np.arange(0.01, 1.00, 0.01):
    threshold = round(float(threshold), 2)
    threshold_pred = threshold_predictions(probabilities, FOCUS_CLASS, threshold)
    threshold_rows.append(
        {
            "threshold": threshold,
            "high_f1": f1_score(
                y_test,
                threshold_pred,
                labels=[FOCUS_CLASS],
                average="macro",
                zero_division=0,
            ),
            "balanced_accuracy": balanced_accuracy_score(y_test, threshold_pred),
        }
    )

threshold_results = pd.DataFrame(threshold_rows).sort_values("high_f1", ascending=False)
threshold_results.head(10)


,threshold,high_f1,balanced_accuracy
75,0.76,0.699868,0.757993
76,0.77,0.699006,0.755667
74,0.75,0.698361,0.759861
73,0.74,0.698147,0.762055
77,0.78,0.697152,0.752774
78,0.79,0.696017,0.749990
72,0.73,0.695825,0.763408
79,0.80,0.695765,0.747938
80,0.81,0.695515,0.745490
71,0.72,0.693884,0.764790


In [8]:
# Final thresholded predictions and report.
best_threshold = threshold_results.iloc[0]["threshold"]
threshold_pred = threshold_predictions(probabilities, FOCUS_CLASS, best_threshold)

print(f"Selected threshold for {FOCUS_CLASS}: {best_threshold:.2f}")
print("\nThreshold classification report:")
print(classification_report(y_test, threshold_pred, digits=4, zero_division=0))
print("Threshold balanced accuracy:", round(balanced_accuracy_score(y_test, threshold_pred), 4))
print("Threshold High F1:", round(f1_score(y_test, threshold_pred, labels=[FOCUS_CLASS], average="macro"), 4))

comparison = pd.DataFrame(
    {
        "prediction_rule": ["default highest probability", f"High threshold >= {best_threshold:.2f}"],
        "high_f1": [
            f1_score(y_test, baseline_pred, labels=[FOCUS_CLASS], average="macro"),
            f1_score(y_test, threshold_pred, labels=[FOCUS_CLASS], average="macro"),
        ],
        "balanced_accuracy": [
            balanced_accuracy_score(y_test, baseline_pred),
            balanced_accuracy_score(y_test, threshold_pred),
        ],
    }
)

comparison


Selected threshold for High: 0.76

Threshold classification report:


              precision    recall  f1-score   support

        High     0.7299    0.6722    0.6999      6303
         Low     0.8745    0.7985    0.8348    110975
      Medium     0.7037    0.8033    0.7502     71722

    accuracy                         0.7961    189000
   macro avg     0.7694    0.7580    0.7616    189000
weighted avg     0.8049    0.7961    0.7982    189000



Threshold balanced accuracy: 0.758


Threshold High F1: 0.6999


,prediction_rule,high_f1,balanced_accuracy
0,default highest probability,0.630326,0.780069
1,High threshold >= 0.76,0.699868,0.757993


## Short discussion

I focused on `High` because it is the rarest irrigation-need class and is also important to catch. I used `High` F1 as the metric and selected a probability threshold of about `0.76`, which had the best held-out `High` F1 in the threshold scan.

Compared with the default highest-probability rule, `High` F1 improved from about `0.63` to about `0.70`. The main tradeoff was that precision improved while recall decreased, meaning the model made fewer false `High` predictions but missed more actual `High` cases. Balanced accuracy moved from about `0.78` to about `0.76`.

Naive Bayes was much weaker than my existing tree/boosting models, which were closer to `0.98` accuracy and weighted F1. Gaussian Naive Bayes is useful as a fast probability baseline, but its independence and normality assumptions are too simple for this dataset.
